## Fill in your name below

**Name:** YOUR NAME HERE

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/antfolk/BMEN35_2026/blob/main/Session4/BMEN35_Ex4_04_Assignment_4_Transformer_experiments.ipynb)

# Assignment 4 — Small changes to text

In Part A, test how a text classifier responds to negation and rewording. In Part B, compare the next-token probabilities for two story prompts, then look at examples sampled from the prompts.

Run the notebook top to bottom on CPU in Colab. Write code in the empty cells in A3 and B3. Before running B3, record your prediction in B2. Write your answers in A4 and B4 using the results printed by your code.

The data, models and helper functions are provided. The walkthroughs **What changes a text prediction?** and **TinyStories: three short text generation experiments** demonstrate the same models. Your code and interpretation are what we assess. The classifier may make mistakes and the clue may raise, lower or leave the probability unchanged.

## Part A. What happens when we change a sentence?

### A1. Load the classifier and try one sentence

The supplied BERT classifier gives a sentence one of two labels. The labels come from SST-2, a sentiment dataset based on movie reviews. Our examples are short, invented device comments. A **reference label** is the label we expect based on the meaning of the sentence. A **predicted label** is the label the model actually chose.

The function `predict_sentiment(text)` returns two values in this order, the predicted label and the score for **positive**. If the model chooses negative with a score of 0.80, the function returns a positive score of 0.20. That gives you a score for the same class in both sentences. Run the practice cell and identify the reference label, predicted label and positive score.

In [ ]:
# Install the library in this notebook's Python environment.
%pip -q install transformers

In [ ]:
from transformers import pipeline                       # Import the helper for tokenization and classification.

classifier = pipeline(                                  # Load the classifier once and reuse it for each sentence.
    'text-classification',                              # Ask for a class label and score for the input text.
    model='Facececersek/bedrock-sentiment-tiny',           # Load the small checkpoint fine-tuned on SST-2.
    device=-1,                                          # Run on CPU.
)


def predict_sentiment(text):                            # Return the label and the score for the positive class.
    result = classifier(text)[0]                        # One input sentence gives one result in the returned list.
    label = result['label'].upper()                     # Read the chosen label and standardise its case.
    winning_score = result['score']                     # This score belongs to whichever class won.

    label_names = {'LABEL_0': 'NEGATIVE', 'LABEL_1': 'POSITIVE'}  # Translate generic names using SST-2's class IDs.
    label = label_names.get(label, label)                # Keep descriptive labels as they are.

    positive_score = winning_score                      # When positive wins, its score is already what we need.
    if label == 'NEGATIVE':                             # When negative wins, calculate the other class's score.
        positive_score = 1 - winning_score              # These two softmax scores add up to one.

    return label, positive_score                         # Give the caller the two values to save for this sentence.

In [ ]:
example = 'The case was sturdy.'                        # Use one practice sentence outside the four pairs.
predicted_label, positive_score = predict_sentiment(example)  # Save the two values from the helper.

print('Sentence:', example)                             # Show what was passed to the model.
print('Reference label: POSITIVE')                       # Show the expected label for the practice sentence.
print('Predicted label:', predicted_label)               # Show the actual label from this run.
print(f'Positive score: {positive_score:.3f}')            # Display the score to three decimal places.

### A2. Read the four pairs

The four pairs below are the input for Part A. Each list entry contains five items in this order: group, sentence A, sentence B, reference label A, reference label B. Treat A as the original and B as the changed sentence. The first two pairs introduce negation and the last two reword the sentence. Read the reference labels to see which changes should flip the label and which should keep it.

In [ ]:
pairs = [                                                # Four tuples, one for each comparison.
    ('Negation',                                        # Pair 1. Inserting not changes the reference label.
     'The instructions were clear.',                    # Sentence A.
     'The instructions were not clear.',                # Sentence B.
     'POSITIVE', 'NEGATIVE'),                            # Reference labels for A and B, in that order.
    ('Negation',                                        # Pair 2. The reference label changes again.
     'The sensor was reliable.',
     'The sensor was not reliable.',
     'POSITIVE', 'NEGATIVE'),
    ('Rewording',                                       # Pair 3. Both sentences express a positive opinion.
     'The controls were easy to use.',
     'I could use the controls without difficulty.',
     'POSITIVE', 'POSITIVE'),
    ('Rewording',                                       # Pair 4. Both sentences express a negative opinion.
     'The alarm was hard to hear.',
     'I had trouble hearing the alarm.',
     'NEGATIVE', 'NEGATIVE'),
]

### A3. Write a loop that prints the four comparisons and totals

First create this dictionary in the empty cell, **before** you start your loop.

```python
correct_by_group = {'Negation': 0, 'Rewording': 0}
```

It stores two separate counts. The key `'Negation'` stores the number of negation pairs with both labels correct. The key `'Rewording'` stores the equivalent number for rewording. Each begins at 0 because we have made no predictions yet. In each iteration, `group` tells you which of the two counts to use.

Loop through all four pairs in their given order. Start with `for group, text_a, text_b, expected_a, expected_b in pairs:`. Inside the loop:

1. Call `predict_sentiment(text_a)` and `predict_sentiment(text_b)`. Save each predicted label and positive score separately.
2. Compare each predicted label with the corresponding reference label. Use `and` to make one Boolean variable, `both_correct`. If that variable is true, add one to the current group's count using `correct_by_group[group] += 1`. If it is false, the count stays as it was.
3. Print the group and one line for each sentence. On each line, include the sentence, its reference label, the predicted label and the positive score to three decimal places. Print the value of `both_correct` once for the pair.

After the loop, print both totals as **Negation: ___ of 2** and **Rewording: ___ of 2**. The total lines go after the loop so that they report the final counts. Your output should show four pair blocks and two totals.

### A4. Use your output to answer two questions

1. Copy your two totals from A3 as **Negation: ___ of 2. Rewording: ___ of 2.**
2. Look at **pair 1**, about the instructions being clear or not clear. State the predicted label and positive score for A and for B. Then say whether the predicted label changed from positive to negative as expected. Also say whether the positive score went down, went up or stayed the same in the printed output. A score can move while the label stays the same.

*A4.1 — Negation: ___ of 2. Rewording: ___ of 2.*

*A4.2 — Pair 1: A was predicted ___ with positive score ___. B was predicted ___ with positive score ___. The expected label change occurred / did not occur. The positive score ___.*

## Part B. Does a clue affect the next token?

### B1. Load TinyStories and try one prompt

A **prompt** is the text we give the model. We can inspect the scores it produces for the **very next token**, and we can also let it sample a longer **continuation** from its scores. These are two different observations from the same model.

The practice cell below shows a sampled continuation. A **seed** sets the starting point for those random draws. When we directly calculate the probability of one next token, there is no random draw and we do not need a seed.

In [ ]:
import torch                                             # PyTorch runs the model and controls the random seed.
from transformers import AutoModelForCausalLM, AutoTokenizer  # Load the language model and its tokenizer.

model_name = 'roneneldan/TinyStories-1M'                  # Choose the model trained to continue short stories.
revision = '2ad005d9b1e727d4a7ed107816c44aff9f78935f'      # Use one saved version for both downloads.
tokenizer = AutoTokenizer.from_pretrained(model_name, revision=revision)  # Load the matching vocabulary.
model = AutoModelForCausalLM.from_pretrained(             # Load the network and its weights.
    model_name, revision=revision, use_safetensors=True,
)
model.eval()                                             # Use prediction behaviour for the network layers.


def continue_story(prompt, seed, temperature=0.8):        # Return only the text the model adds.
    torch.manual_seed(seed)                             # Reset the random starting point for this call.
    inputs = tokenizer(prompt, return_tensors='pt')       # Convert the prompt to tensors of token IDs and input information.
    prompt_length = inputs['input_ids'].shape[1]          # Count the input tokens along the sequence axis.

    with torch.inference_mode():                         # Run prediction without gradients for training.
        generated_ids = model.generate(                  # Choose and append one token at a time.
            **inputs,                                    # Include the tokenizer's named inputs.
            max_new_tokens=24,                           # Limit the continuation to at most 24 extra tokens.
            do_sample=True,                              # Draw tokens according to their probabilities.
            num_beams=1,                                 # Follow one continuation for each call.
            temperature=temperature,                     # Use 0.8 for both conditions in the assignment.
            top_k=20,                                    # Sample from the 20 highest scoring choices at each step.
            pad_token_id=tokenizer.eos_token_id,          # Use this model's end token as padding if needed.
        )

    continuation_ids = generated_ids[0, prompt_length:]  # Skip the IDs that were already in the prompt.
    return tokenizer.decode(continuation_ids, skip_special_tokens=True)  # Turn only the new IDs into text.

In [ ]:
practice_prompt = 'Once upon a time, Leo found a'        # Practise with a different story opening.
practice_story = continue_story(practice_prompt, seed=7)  # Use seed 7 and the default temperature.
print('Prompt:', practice_prompt)                         # Show the given input.
print('Continuation:', repr(practice_story))              # Show the new text, with its spaces visible.

### B2. Read the prompts and record your prediction

We use two prompts that both end with **Inside, she saw a**. One adds a sentence saying Mia heard a cat meow. This gives the model a clue. Look at the **five most probable next tokens** for each prompt. We calculate these probabilities over the whole vocabulary, before temperature and top-k filtering.

Rank 1 is the model's most probable next token. The token `' cat'` (including its space) might be missing from one or both top-five lists. The two lines below the table always show its probability and rank in the full vocabulary. A higher probability means the model gives the token more weight for that prompt. A rank closer to 1 means it is placed higher among the alternatives. The table does not say which token will be chosen in a particular random draw.

Before running B3, predict whether the clue will give `' cat'` a higher, lower or equal probability compared with the original prompt. Write one sentence and keep both prompts as provided.

*My prediction is ___.*

In [ ]:
prompt_without_cue = 'Mia opened the box. Inside, she saw a'   # No clue about the contents.
prompt_with_cue = 'Mia opened the box. She heard a cat meow. Inside, she saw a'  # Add one clue sentence.
target_piece = ' cat'                                           # The space belongs to the token.
seeds = [7, 13, 21]                                              # Use each seed for both sampled continuations.


def next_token_probabilities(prompt):                           # Return one probability per vocabulary ID.
    inputs = tokenizer(prompt, return_tensors='pt')              # Encode the prompt as PyTorch tensors.
    with torch.inference_mode():                                # Run one prediction without sampling or training.
        output = model(**inputs)                                # Score the possible next tokens at every input position.
    next_logits = output.logits[0, -1]                           # Choose the only prompt and its last input position.
    return torch.softmax(next_logits, dim=0)                     # Normalise the scores over the whole vocabulary.


def show_next_token_comparison(prompt_without, prompt_with, piece):
    without = next_token_probabilities(prompt_without)           # Run the model on the original prompt.
    with_cue = next_token_probabilities(prompt_with)              # Run the same model on the prompt with the clue.

    top_without, ids_without = torch.topk(without, k=5)           # Get the five highest probabilities and their IDs.
    top_with, ids_with = torch.topk(with_cue, k=5)                  # Find the top five for the other prompt separately.

    print(f'{"Rank":>4} | {"Without cue":<26} | With cue')          # Read the two candidate lists side by side.
    print('-' * 58)                                              # Separate the header from the candidate rows.
    for index in range(5):                                       # Read the candidates in order from highest to lowest.
        piece_without = tokenizer.decode([ids_without[index].item()])  # Turn this ID into a readable token.
        piece_with = tokenizer.decode([ids_with[index].item()])   # Read the token of the same rank in the other list.
        left = f'{piece_without!r} ({top_without[index].item():.4%})'  # Show the space and probability.
        right = f'{piece_with!r} ({top_with[index].item():.4%})'   # Use the same format for the cue condition.
        print(f'{index + 1:>4} | {left:<26} | {right}')            # Rank 1 is the most probable next token.

    target_ids = tokenizer.encode(piece, add_special_tokens=False)  # Look up the exact target piece, including its space.
    if len(target_ids) != 1:                                      # This experiment compares one token.
        raise ValueError('The target must be one token for this tokenizer.')
    target_id = target_ids[0]                                     # Select the vocabulary index of the target.

    p_without = without[target_id].item()                       # Find the target probability even if it is outside the top five.
    p_with = with_cue[target_id].item()                           # Find the same token in the cue distribution.
    rank_without = int((without > p_without).sum().item()) + 1   # Count the candidates ahead of it, then add one.
    rank_with = int((with_cue > p_with).sum().item()) + 1           # Do the same in the second distribution.

    print(f'\nTarget {piece!r} without cue: {p_without:.4%}, rank {rank_without}')
    print(f'Target {piece!r} with cue:    {p_with:.4%}, rank {rank_with}')

    return p_without, p_with, rank_without, rank_with             # Return these numbers for the follow-up comparison.

### B3. Print the top-five table, then sample continuations

**First, show the next-token probabilities.** Start your code cell with:

```python
show_next_token_comparison(prompt_without_cue, prompt_with_cue, target_piece)
```

The provided function prints the two top-five lists side by side. Each row shows the token at that rank and its probability for each prompt. The two lines below the table report the probability and rank of `' cat'` for both prompts. This comparison uses no random seed.

**Then, sample examples to inspect.** Write a loop over `seeds`. For each seed, call `continue_story(prompt_without_cue, seed)` and `continue_story(prompt_with_cue, seed)`. Keep the default temperature in both calls. Print the seed followed by the two full continuations and label which prompt produced each one.

The output should contain the top-five table, the two lines for `' cat'` and three pairs of sampled continuations.

### B4. Compare the candidate table and one pair of stories

1. Copy the probability and rank for `' cat'` from each of the two lines below the table. State whether `' cat'` appears in the top five for neither prompt, one prompt or both. Say whether its probability went up, went down or stayed the same after adding the clue, and whether that agrees with your prediction in B2.
2. Read the two continuations at **seed 7**. Quote the first few generated words for each, leaving out the prompt. Say whether either begins with **cat**. Explain in one sentence why a token can move up in the candidate table and still be absent from a sampled continuation.

*B4.1 — Without cue: ___%, rank ___. With cue: ___%, rank ___. In the top five: ___. The probability went ___. This agrees / does not agree with my prediction because ___.*

*B4.2 — For seed 7 the two continuations began “___” and “___”. Which, if any, starts with cat: ___. A higher-ranked token can still be absent from a sampled continuation because ___.*

## Submit your notebook

1. Fill in your name at the top.
2. Check that you have code and visible output for A3 and B3, a prediction in B2, and written answers to both questions in A4 and both questions in B4.
3. In Google Colab, choose **File → Download → Download .ipynb** and submit the downloaded `.ipynb` file in Canvas.